# LaGoVAD-KIM — arm **K5ln**: per-step LayerNorm on the motion columns (seed 2024)

Rule (written before any number): `core/docs/final/KIM_RUN.md` §25. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad`. Nothing is extracted.

**K5ln = K5 s2024 + `model.motion_norm=layernorm`**: the baked motion columns `c·(u − m_u)/σ_u` get a per-step LayerNorm (no affine) and are rescaled by the cache's `c`, then `W_u` as before. The config differs from K5 in that one key. **The control is K5 s2024**, already on Drive with its reads: it is not retrained.

| step | what | GPU | estimate |
|---|---|---|---|
| 1–3 | setup, preflight, pytest (**raises if red**) | no | 5–15 min |
| 4 | train **K5ln** (skips if verified on Drive); J10, config diff, `norm_scale == c`, start check vs K5 | **yes** | 15–45 min |
| 5 | read K5ln on DoTA-CAP-dev (protocol B), DADA (whole), TAD | **yes** | 10–20 min |
| 6 | read-out → paste `REPORTS/kim_readout_k5ln_s2024.md` to Claude; the `ln_rule` line is §25's decision | no | 2–5 min |

Every step skips what is already on Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window, A2 bakes, KNN, v2 A2 runs
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code + every output this notebook writes
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert 'decided control' in getattr(constants, 'KIM_PILOT_DISCLOSURE', ''), 'this upload predates the pilot move -- re-upload kat-vad/'
assert '### 6.3 Kcap' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §6.3 in this upload -- re-upload kat-vad/'
assert 'motion_norm' in (REPO / 'core/config.py').read_text() and 'ln_rule' in (REPO / 'core/tools/kim_readout.py').read_text(), 'this upload has no arm K5ln -- re-upload kat-vad/'
assert '## 25.' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §25 in this upload -- re-upload kat-vad/'

SEED     = constants.KIM_SEED                      # 2024; KIM_RUN.md §25
ENC      = constants.V5_ENCODER                    # V2-S
C        = constants.NEXAR_CONSTRUCTION_WINDOW
DATASET  = constants.NEXAR_DATASET
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
W0       = constants.ZS_CONTROL_ARM                # 'W0'
K0, K0K9, K2, K5 = constants.KIM_ARMS              # ('K0', 'K0k9', 'K2', 'K5')
KCAP     = constants.KIM_CAP_ARM
K5D      = constants.KIM_DVS_ARM                   # W0 + K5 + seam mask, DVS on (§12)
K5LN     = constants.KIM_LN_ARM                    # K5 + model.motion_norm=layernorm (§25)
ARMS     = (K5, K5LN)                              # K5 = the control, on Drive; only K5ln trains

V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
BUILD    = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n4' / 'build_report.json'
DATA     = Path(DRIVE_V2) / 'data' / f'{DATASET}_{C}'
A2_NEXAR = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'
A2_DADA  = V2_CACHE / 'v2' / 'A2_S' / f'{T2NAME}_test_sources_nexar'
A2_DOTA  = V2_CACHE / 'v2' / 'A2_S' / f'{CAP_DEV}_s3_from_s1_nexar'
A2_TAD   = V2_CACHE / 'v2' / 'A2_S' / f'{constants.TAD_DATASET}_s8_nexar'
KNN      = V2_CACHE / 'knn' / f'{DATASET}_{C}' / constants.KNN_CACHE_FILENAME
T2       = constants.DATA_ROOT / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
TAD      = constants.DATA_ROOT / constants.TAD_DATASET
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
ZS_ROOT  = constants.OUTPUT_ROOT / constants.ZS_OUTPUT_DIRNAME
A2_RUNS  = V2_OUT / constants.NEXAR_FULL_RUNS_DIR / 'A2'
# W0 = v2 Nexar A2 s2024, same code path and seed (ZS_TAD_DADA.md §2: DoTA-CAP-dev 0.8110072 in both).
# Both checkpoints are gone (KIM_RUN.md §4 dated note): W0 survives as a config (the K-rows' config diff) and
# as saved reads (a printed row). Nothing here loads W0's weights.
W0_SEED  = constants.KIM_SEED                      # W0's reads and config exist for this seed only
W0_CONFIGS = [constants.OUTPUT_ROOT / constants.WU_OUTPUT_DIRNAME / W0 / f's{W0_SEED}' / 'config.yaml', A2_RUNS / f's{W0_SEED}' / 'config.yaml']
W0_CONFIG = next((p for p in W0_CONFIGS if p.exists()), None)
assert W0_CONFIG is not None, f'no W0 config: upload the repo\'s outputs/v_final/wu/W0/s{W0_SEED}/config.yaml to {W0_CONFIGS[0]}'
print(f'W0 config: {W0_CONFIG}')
W0_READ = {'dota': constants.OUTPUT_ROOT / constants.WU_OUTPUT_DIRNAME / 'eval' / 'dota' / CAP_DEV / W0,
           'dada': ZS_ROOT / 'eval' / 'dada' / W0,
           'tad':  ZS_ROOT / 'eval' / 'tad' / W0}            # holds s2024/
ROOT     = constants.OUTPUT_ROOT / constants.KIM_OUTPUT_DIRNAME    # Thesis-Final/outputs/kim
REPORTS  = ROOT / 'REPORTS'
RUN_DIR  = {a: ROOT / a / f's{SEED}' for a in (*constants.KIM_ARMS, KCAP, K5D, K5LN)}
def eval_root(seed):
    return ROOT / ('eval' if seed == W0_SEED else f'eval_s{seed}')    # s2024 keeps kim_runs' dirs


EVAL     = {s: eval_root(SEED) / s for s in ('dota', 'dada', 'tad')}
LOCAL    = Path('/content/kim')
for p in (REPORTS, LOCAL):
    p.mkdir(parents=True, exist_ok=True)

PLAN = json.loads(BUILD.read_text())['plan'][C]
STEPS, EPOCHS = PLAN['steps'], PLAN['epochs']     # 1,749 = 33 x 53
for name, path in (('Nexar_window', DATA), ('Nexar A2 bake', A2_NEXAR), ('DADA A2 bake', A2_DADA),
                   ('DoTA-CAP A2 bake', A2_DOTA), ('TAD A2 bake', A2_TAD), ('KNN', KNN),
                   ('T2 meta', T2 / constants.META_FILENAME), ('T2 census', T2_CENSUS), ('DADA annotation', ANNOT),
                   ('TAD labels', TAD / constants.FRAME_LABELS_TEST_FILENAME), ('DoTA s1', DOTA_S1)):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:18s} {path}')
print(f'steps {STEPS} ({EPOCHS} epochs) | runs/reads -> {ROOT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn, openpyxl; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 5-9 need one'
df -h /content | tail -1

In [ ]:
import pickle
import shutil
import subprocess
import time

import numpy as np
import torch
import yaml

from core.config import Config                       # noqa: E402
from core.data.v2_splits import load_split           # noqa: E402
from core.tools.dada_whole_eval import test_sources  # noqa: E402
from core.tools.v5_stage1 import _flat               # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    '''Copy a Drive dir (only ``ids`` + json manifests if given) or file to VM disk once (bit-identical; C2).'''
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    if ids is None:
        for f in src.iterdir():
            if f.is_file() and not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
    else:
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    '''VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    t0 = time.time()
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print(f'Drive flushed and remounted in {time.time() - t0:.0f}s')


def ckpt_step(path):
    '''The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step).'''
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def is_verified(drive_dir):
    rec, ckpt = drive_dir / 'verified.json', drive_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == STEPS and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


# ids and the preflight: nothing below runs on a wrong corpus
DADA_IDS = test_sources(T2)
CAP_IDS = sorted(load_split(CAP_DEV))
TAD_IDS = sorted(json.loads((TAD / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
assert len(DADA_IDS) == 370, len(DADA_IDS)
assert len(TAD_IDS) == constants.V2_TADZ_VIDEOS, len(TAD_IDS)
for name, d, ids in (('DADA', A2_DADA, DADA_IDS), ('DoTA-CAP', A2_DOTA, CAP_IDS), ('TAD', A2_TAD, TAD_IDS)):
    miss = [v for v in ids if not (d / f'{v}.npy').exists()]
    assert not miss, f'{len(miss)} {name} videos have no A2 bake: {miss[:5]}'
for name, d in (('Nexar', A2_NEXAR), ('DADA', A2_DADA), ('DoTA-CAP', A2_DOTA), ('TAD', A2_TAD)):
    m = json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
    assert (m['crn'], m['motion']) == (constants.V2_OFF, ENC), f'{name} bake is {m}'
same = {json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())['train_ids_sha1']
        for d in (A2_NEXAR, A2_DADA, A2_DOTA, A2_TAD)}
assert len(same) == 1, f'the bakes carry different statistics: {same}'
w0_cfg = _flat(yaml.safe_load(W0_CONFIG.read_text()))
assert w0_cfg['model.score_head_kernel'] == constants.DADA_ORIGIN_SCORE_HEAD_KERNEL, w0_cfg['model.score_head_kernel']
assert w0_cfg['dvs.theta'] < 1 and not w0_cfg.get('kim.enabled', False), 'W0 is not the DVS-on host'
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in DATA.glob('*.json'):               # defs.json etc.: the definitions the arms trained with
    shutil.copy2(f, L_DEFS / f.name)
print(f'preflight OK: {len(DADA_IDS)} DADA, {len(CAP_IDS)} DoTA-CAP-dev, {len(TAD_IDS)} TAD videos; shared statistics')

## 3. Tests on the KIM code (CPU) — **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_kim.py',
     'core/tests/test_kim_tools.py', 'core/tests/test_wu_arms.py', 'core/tests/test_motion_norm.py'])

## W0's saved reads, K5's saved run and reads, and the read helper

In [ ]:
L_T2 = LOCAL / 'T2_meta'
L_T2.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2 / constants.META_FILENAME)
L_TAD = LOCAL / 'TAD'
L_TAD.mkdir(parents=True, exist_ok=True)
for f in TAD.iterdir():                     # json / txt only -- never a frame folder
    if f.is_file() and f.suffix in ('.json', '.txt'):
        shutil.copy2(f, L_TAD / f.name)


def read_arm(arm):
    # one arm's reads with W0's protocols, into outputs/kim/eval/<set>/<arm>; skips what exists
    ckpt = RUN_DIR[arm] / 'checkpoint_last.pt'
    wrote = False
    if not (EVAL['dota'] / arm / 'clip_aucs.json').exists():
        local = LOCAL / 'dota' / f's{SEED}' / arm
        run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', ckpt,
             '--input-dir', stage(A2_DOTA, LOCAL / 'bake_dota', CAP_IDS), '--s1-dir', stage(DOTA_S1, LOCAL / 'dota_s1', CAP_IDS),
             '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
             '--data-dir', stage(DOTA / 'labels_s8', LOCAL / 'dota_labels'), '--split', CAP_DEV,
             '--out-dir', local, '--device', 'auto'], log=LOCAL / f'dota_{arm}.log')
        publish(local, EVAL['dota'] / arm)
        wrote = True
    if not (EVAL['dada'] / arm / 'clip_aucs.json').exists():
        local = LOCAL / 'dada' / f's{SEED}' / arm
        run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--run', f's{SEED}', ckpt, '--t2-dir', L_T2,
             '--annotation', ANNOT, '--census', T2_CENSUS, '--input-dir', stage(A2_DADA, LOCAL / 'bake_dada', DADA_IDS),
             '--data-dir', L_DEFS, '--out-dir', local], log=LOCAL / f'dada_{arm}.log')
        publish(local, EVAL['dada'] / arm)
        wrote = True
    tad_out = EVAL['tad'] / arm / f's{SEED}'
    if not (tad_out / 'results.json').exists():
        local = LOCAL / 'tad' / arm / f's{SEED}'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.evaluate', '--ckpt', ckpt, '--set', f'data.dataset={constants.TAD_DATASET}',
             '--data-dir', L_TAD, '--clip-dir', stage(A2_TAD, LOCAL / 'bake_tad', TAD_IDS),
             '--score-norm', 'auto', '--save-scores', '--output-dir', local], log=LOCAL / f'tad_{arm}.log')
        publish(local, tad_out)
        wrote = True
    if wrote:
        flush_drive()
    dota = json.loads((EVAL['dota'] / arm / 'protocol_b_readout.json').read_text())
    dada = json.loads((EVAL['dada'] / arm / 'dada_readout.json').read_text())
    tad = json.loads((tad_out / 'results.json').read_text())
    print(f"{arm}: DoTA-CAP-dev macro {dota['macro']['mean']:.4f} | DADA macro {dada['macro']['mean']:.4f} "
          f"| TAD macro {tad['auc_macro']:.4f}")


W0_NEEDS = {'dota': ('clip_aucs.json', 'protocol_b_readout.json'), 'dada': ('clip_aucs.json', 'dada_readout.json')}
missing = [W0_READ[s] / f for s, files in W0_NEEDS.items() for f in files if not (W0_READ[s] / f).exists()]
tad_scores = W0_READ['tad'] / f's{W0_SEED}' / 'scores'
n_tad = len(list(tad_scores.glob('*.npz'))) if tad_scores.is_dir() else 0
if n_tad != constants.V2_TADZ_VIDEOS:          # C10: a folder is not evidence of its files
    missing.append(tad_scores)
assert not missing, ("W0's reads cannot be redone (no checkpoint). Upload from the repo's outputs/v_final/ to the same "
                     f"path under {constants.OUTPUT_ROOT}: {[str(p.relative_to(constants.OUTPUT_ROOT)) for p in missing]}")
print(f'W0 reads OK: {W0_READ}')


K5_NEEDS = [RUN_DIR[K5] / 'config.yaml', RUN_DIR[K5] / 'metrics.jsonl',
            EVAL['dota'] / K5 / 'clip_aucs.json', EVAL['dada'] / K5 / 'clip_aucs.json',
            EVAL['tad'] / K5 / f's{SEED}' / 'results.json']
missing = [p for p in K5_NEEDS if not p.exists()]
assert not missing, f'K5 s{SEED} (the control) is incomplete on Drive: {missing}'
k5_cfg = yaml.safe_load((RUN_DIR[K5] / 'config.yaml').read_text())
assert k5_cfg['kim']['edge_mask'] is True and k5_cfg['kim']['prototypes'] is True, 'K5 s2024 is not the main-model recipe'
assert k5_cfg['model'].get('motion_norm', constants.MOTION_NORM_OFF) == constants.MOTION_NORM_OFF, 'K5 already has a motion norm?'
print(f'K5 s{SEED} run + reads OK')


## 4. Train K5ln (GPU)

Watch the log: past 2 s/step, stop and paste it. Then J10, the config diff against **K5** (only `model.motion_norm` may differ, plus keys added after K5 trained at their defaults), `norm_scale` == the Nexar bake's `c`, and §25's start check: step-1 `mil` within `KIM_START_TOL` of K5's (`W_u` = 0 at step 1, so the forward is identical). A mismatch **flags** the contrast; it does not stop the run.

In [ ]:
DEFAULTS = _flat(Config().to_dict())


def arm_sets(arm, kernel):
    extra = {K0: [], K2: ['kim.enabled=true', 'kim.edge_mask=true'],                            # §5.2, §5.3; only after §6.2 reads reproduced
             K5: ['kim.enabled=true', 'kim.prototypes=true', 'kim.edge_mask=true'],             # §5.3
             KCAP: ['kim.enabled=true', 'kim.prototypes=true', 'kim.capacity_control=true'],    # §6.3
             K5D: ['kim.enabled=true', 'kim.prototypes=true', 'kim.edge_mask=true', 'kim.seam_mask=true'],  # §12
             K5LN: ['kim.enabled=true', 'kim.prototypes=true', 'kim.edge_mask=true',
                    f'model.motion_norm={constants.MOTION_NORM_LAYERNORM}']}[arm]                      # §25
    dvs = [] if arm == K5D else ['dvs.theta=1.0', 'dvs.theta_ego=1.0']                          # D2; K5d keeps W0's DVS
    return [f'model.score_head_kernel={kernel}', *dvs, *extra]


def train_cmd(arm, kernel, out_dir, data, knn):
    cmd = [sys.executable, '-m', 'core.train',
           '--set', 'train.stage=2', '--set', f'train.seed={SEED}',
           '--set', f'train.num_epochs={EPOCHS}', '--set', 'train.amp=true',
           '--set', f'data.dataset={DATASET}',
           '--set', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
           '--set', 'kip.enabled=false', '--set', 'loss.dvs_anchor_mode=span',
           '--set', 'data.is_egocentric=false',
           '--set', f'v2.motion={ENC}', '--set', f'model.motion_dim={constants.V5_GRID_DIM}']
    for s in arm_sets(arm, kernel):
        cmd += ['--set', s]
    return cmd + ['--data-dir', data, '--clip-dir', A2_LOCAL, '--output-dir', out_dir, '--knn-cache', knn]


def train_arm(arm, kernel):
    drive_dir, local = RUN_DIR[arm], LOCAL / 'runs' / f's{SEED}' / arm    # never resume another seed's run
    if is_verified(drive_dir):
        print(f'{arm} s{SEED}: verified on Drive -- not retrained')
        return
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    t0 = time.time()
    run(train_cmd(arm, kernel, local, DATA_LOCAL, KNN_LOCAL) + resume, log=LOCAL / f'train_{arm}.log')
    last = metrics_last(local / 'metrics.jsonl')
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{arm}: J10 locally'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        publish(local / name, drive_dir / name)
    flush_drive()
    assert ckpt_step(drive_dir / 'checkpoint_last.pt') == STEPS, f'{arm}: Drive copy is not step {STEPS}'
    (LOCAL / 'verified.json').write_text(json.dumps(
        {'step': STEPS, 'ckpt_bytes': (drive_dir / 'checkpoint_last.pt').stat().st_size}))
    publish(LOCAL / 'verified.json', drive_dir / 'verified.json')
    print(f'{arm}: {(time.time() - t0) / 60:.0f} min | mil {last.get("mil")} | kim_mil {last.get("kim_mil")} '
          f'| W_out {last.get("kim_w_out_norm")} | proto cos {last.get("kim_proto_cos")}')


def check_run(arm, kernel):
    d = RUN_DIR[arm]
    payload = torch.load(d / 'checkpoint_last.pt', map_location='cpu', weights_only=False)
    assert payload['global_step'] == metrics_last(d / 'metrics.jsonl')['global_step'] == STEPS, f'{arm}: J10'
    got = _flat(yaml.safe_load((d / 'config.yaml').read_text()))
    ref = _flat(k5_cfg)                       # §25: the config differs from K5 in model.motion_norm alone
    allowed = {'model.motion_norm'}
    diff = {}
    for k in set(ref) | set(got):
        if ref.get(k) == got.get(k) or k in allowed:
            continue
        if k not in ref and got.get(k) == DEFAULTS.get(k):
            continue                      # a key added after K5 trained, left at its default
        diff[k] = (ref.get(k), got.get(k))
    assert not diff, f'{arm} differs from K5 beyond model.motion_norm: {diff}'
    assert got['model.motion_norm'] == constants.MOTION_NORM_LAYERNORM, got['model.motion_norm']
    c = json.loads((A2_NEXAR / constants.V2_INPUT_MANIFEST_FILENAME).read_text())['c']
    scale = float(payload['model']['motion_residual.norm_scale'])
    assert abs(scale - c) <= 1e-6 * c, f'{arm}: norm_scale {scale} != the Nexar bake c {c}'
    assert len(payload['optimizer']['param_groups']) == 1, f'{arm}: one AdamW group (D13)'
    first = {a: json.loads((RUN_DIR[a] / 'metrics.jsonl').read_text().splitlines()[0]) for a in (arm, K5)}
    gap = abs(first[arm]['mil'] - first[K5]['mil'])
    same = gap <= constants.KIM_START_TOL
    print(f'{arm}: J10 {STEPS}, config == K5 + motion_norm, norm_scale {scale:.6f} == c | step-1 mil '
          f'{first[arm]["mil"]:.6f} vs K5 {first[K5]["mil"]:.6f} (gap {gap:.1e}): '
          f'{"same start" if same else "START MISMATCH: the read-out carries this flag"}')
    print(f'torch {torch.__version__} | {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU"}')


KERNEL = constants.DADA_ORIGIN_SCORE_HEAD_KERNEL     # §3.1: the host keeps kernel 3
TO_TRAIN = [arm for arm in ARMS if not is_verified(RUN_DIR[arm])]
assert K5 not in TO_TRAIN, 'K5 s2024 is the control and must be verified on Drive; it is never retrained here'
print(f'seed {SEED}: to train {TO_TRAIN}, on Drive {[a for a in ARMS if a not in TO_TRAIN]}')
if TO_TRAIN:                                         # C39: nothing is staged for a finished run
    A2_LOCAL = stage(A2_NEXAR, LOCAL / 'stage' / 'a2_nexar')
    DATA_LOCAL = stage(DATA, LOCAL / 'stage' / 'data')
    KNN_LOCAL = stage(KNN, LOCAL / 'stage' / 'knn.npz')
    for arm in TO_TRAIN:
        train_arm(arm, KERNEL)
        check_run(arm, KERNEL)


## 5. Zero-shot reads (GPU): DoTA-CAP-dev protocol B, DADA-2000 whole video, TAD

In [ ]:
for arm in ARMS:
    if not is_verified(RUN_DIR[arm]):
        raise RuntimeError(f'{arm} s{SEED} is not on Drive')
    read_arm(arm)                  # K5's reads exist and are skipped


## 6. Read-out → paste `REPORTS/kim_readout_k5ln_s2024.md` to Claude

Prints **K5ln − K5** (paired per video, the §25 contrast) and K5ln − W0, the start check, and the per-step norm of the baked motion columns on each corpus (what the LayerNorm erases; printed, never decides). The **`ln_rule`** line is §25's decision.

In [ ]:
def readout_ln():
    root = eval_root(SEED)
    cmd = [sys.executable, '-m', 'core.tools.kim_readout', '--control', K5, '--seed', SEED,
           '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
           '--dota', W0, W0_READ['dota'], '--dada', W0, W0_READ['dada'], '--tad', W0, W0_READ['tad']]
    for arm in ARMS:
        cmd += ['--dota', arm, root / 'dota' / arm, '--dada', arm, root / 'dada' / arm,
                '--tad', arm, root / 'tad' / arm, '--run', arm, ROOT / arm / f's{SEED}']
    for name, src, dst, ids in (('nexar (all baked ids)', A2_NEXAR, LOCAL / 'stage' / 'a2_nexar', None),
                                ('DoTA-CAP-dev', A2_DOTA, LOCAL / 'bake_dota', CAP_IDS),
                                ('DADA', A2_DADA, LOCAL / 'bake_dada', DADA_IDS),
                                ('TAD', A2_TAD, LOCAL / 'bake_tad', TAD_IDS)):
        cmd += ['--motion-cache', name, stage(src, dst, ids)]
    name = f'kim_readout_k5ln_s{SEED}'
    local = LOCAL / name
    run(cmd + ['--out-dir', local])
    publish(local / 'kim_readout.md', REPORTS / f'{name}.md')
    publish(local / 'kim_readout.json', REPORTS / f'{name}.json')
    print((REPORTS / f'{name}.md').read_text())


readout_ln()
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
drive.mount('/content/drive')
print(f'Drive flushed -- paste REPORTS/kim_readout_k5ln_s{SEED}.md to Claude')
